In [1]:
# ==============================================================================
# ПРАКТИКА 9 (Варіант 6: Чернігів)
# ==============================================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. Побудова набору клієнтів
# ------------------------------------------------------------------------------
np.random.seed(42)
base_price = 690  # Базова ціна для Варіанта 6 (Чернігів)

df = pd.DataFrame({
    "клієнт_id": range(1, 11),
    "вік": [21, 45, 29, 58, 33, 62, 24, 50, 38, 27],
    "дохід": [18000, 45000, 28000, 60000, 32000, 52000, 21000, 48000, 35000, 26000],
    "кількість": [2, 5, 3, 6, 4, 2, 8, 4, 5, 3],
    "місто_доставки": ["Чернігів", "Київ", "Львів", "Чернігів", "Одеса", "Київ", "Чернігів", "Львів", "Одеса", "Чернігів"]
})

print("=== Завдання 1: Початковий набір даних клієнтів ===")
print(df)
print("\n" + "="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Похідні змінні
# ------------------------------------------------------------------------------
df["сума_замовлення"] = base_price * df["кількість"]
df["частка_доходу"] = df["сума_замовлення"] / df["дохід"]

print("=== Завдання 2: Додавання похідних змінних ===")
print(df[["клієнт_id", "дохід", "сума_замовлення", "частка_доходу"]])

print("\n[Висновок Завдання 2]:")
print("Змінна 'частка_доходу' показує фінансове навантаження покупки на бюджет клієнта.")
print("Абсолютна сума замовлення в 3450 грн для людини з доходом 18 000 грн складає ~19% бюджету (відчутна покупка),")
print("а для людини з доходом 60 000 грн — лише ~5.7% (дрібна покупка). Це дає кращу сегментацію, ніж сума і дохід окремо.\n")
print("="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. Дискретизація
# ------------------------------------------------------------------------------
df["вік_cut"] = pd.cut(df["вік"], bins=3)
df["вік_qcut"] = pd.qcut(df["вік"], q=3)

print("=== Завдання 3: Дискретизація (pd.cut vs pd.qcut) ===")
print("Розподіл pd.cut() (рівні інтервали за значеннями):")
print(df["вік_cut"].value_counts().sort_index())

print("\nРозподіл pd.qcut() (рівна кількість об'єктів у групах):")
print(df["вік_qcut"].value_counts().sort_index())

print("\n[Висновок Завдання 3]:")
print("Кількість клієнтів у групах відрізняється, тому що pd.cut() ділить діапазон значень (від Min до Max) на 3 рівні за шириною інтервали,")
print("і кількість попадань залежить від щільності даних. Зато pd.qcut() підбирає межі так, щоб у кожній групі опинилася однакова кількість клієнтів (~3-4).\n")
print("="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Масштабування
# ------------------------------------------------------------------------------
# Min-Max нормалізація в [0, 1]
df["вік_minmax"] = (df["вік"] - df["вік"].min()) / (df["вік"].max() - df["вік"].min())
df["дохід_minmax"] = (df["дохід"] - df["дохід"].min()) / (df["дохід"].max() - df["дохід"].min())

# Z-score стандартизація
df["вік_z"] = (df["вік"] - df["вік"].mean()) / df["вік"].std()
df["дохід_z"] = (df["дохід"] - df["дохід"].mean()) / df["дохід"].std()

print("=== Завдання 4: Результати масштабування ===")
print(df[["вік", "дохід", "вік_minmax", "дохід_minmax", "вік_z", "дохід_z"]].head())

print("\n[Висновок Завдання 4]:")
print("До масштабування дохід вимірювався в десятках тисяч (18000-60000), а вік — у десятках (21-62).")
print("При розрахунку дистанцій (наприклад, у k-NN чи k-means) дохід би повністю домінував над віком лише через свій масштаб.")
print("Масштабування приводить обидві змінні до єдиної шкали без спотворення їхнього внутрішнього розподілу.\n")
print("="*80 + "\n")


# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. Індикаторне кодування (One-Hot Encoding)
# ------------------------------------------------------------------------------
city_dummies = pd.get_dummies(df["місто_доставки"], prefix="місто", dtype=int)
df_final = pd.concat([df, city_dummies], axis=1)

print("=== Завдання 5: Індикаторне кодування міст ===")
print(df_final[["місто_доставки"] + list(city_dummies.columns)].head())

print("\n[Висновок Завдання 5]:")
print(f"Створено {city_dummies.shape[1]} нових стовпців, що дорівнює кількості унікальних міст у даних ({df['місто_доставки'].nunique()}).")
print("Для 'місто_доставки' доречний саме One-Hot, бо це номінальна категорія без природного порядку.")
print("Порядкове кодування (1, 2, 3...) внісло б хибну ієрархію (ніби 'Львів' утричі більший чи важливіший за 'Чернігів').\n")
print("="*80 + "\n")


# ------------------------------------------------------------------------------
# ВІДПОВІДІ НА КОНТРОЛЬНІ ПИТАННЯ
# ------------------------------------------------------------------------------
print("""=== КОНТРОЛЬНІ ПИТАННЯ ===

1. Різниця між pd.cut() і pd.qcut():
   - pd.cut() ділить діапазон на інтервали однакової довжини (ширини). Кількість елементів у групах може бути різною.
   - pd.qcut() ділить дані за квантилями, гарантуючи однакову (або максимально близьку) кількість елементів у кожній групі.

2. Чому One-Hot, а не порядкове кодування для міст:
   - Міста є номінальною категорією (назви без порядку). Якщо надати їм коди 1, 2, 3, математичні моделі сприйматимуть 
     це як числовий порядок (3 > 1), що призведе до некоректних обчислень та хибних висновків.

3. Навіщо масштабувати змінні перед чутливими методами:
   - Алгоритми на основі відстаней (k-NN, k-means) або регресії з регуляризацією вважають змінні з більшим числовим 
     діапазоном важливішими. Масштабування зрівнює вклад усіх ознак у фінальний результат.

4. Чому похідна змінна (частка доходу) часто корисніша:
   - Вона комбінує декілька вихідних параметрів в один відносний показник. Це дає бізнес-контекст, 
     який нормалізує абсолютні цифри та дозволяє порівнювати клієнтів з різними рівнями доходу.
""")

=== Завдання 1: Початковий набір даних клієнтів ===
   клієнт_id  вік  дохід  кількість місто_доставки
0          1   21  18000          2       Чернігів
1          2   45  45000          5           Київ
2          3   29  28000          3          Львів
3          4   58  60000          6       Чернігів
4          5   33  32000          4          Одеса
5          6   62  52000          2           Київ
6          7   24  21000          8       Чернігів
7          8   50  48000          4          Львів
8          9   38  35000          5          Одеса
9         10   27  26000          3       Чернігів


=== Завдання 2: Додавання похідних змінних ===
   клієнт_id  дохід  сума_замовлення  частка_доходу
0          1  18000             1380       0.076667
1          2  45000             3450       0.076667
2          3  28000             2070       0.073929
3          4  60000             4140       0.069000
4          5  32000             2760       0.086250
5          6  52000       